# Milestone 5 Lab, notebook 5 of 7: Crash and carry on

## Where we are

| Notebook | What you do |
|---|---|
| 1. Files arrive, by hand | Load the files the naive way, and watch three things go wrong |
| 2. A loader that is safe to repeat | Fingerprint each file, check its columns, and choose a rule for every file |
| 3. AI reads a changed file | Ask a model what a renamed column means, and score its suggestions |
| 4. A workflow that can wait | Build the loader as a graph that pauses for your approval |
| **5. Crash and carry on** (you are here) | Restart the kernel on purpose, and find the file still waiting |
| 6. A swarm of file workers | Load fifty files side by side, and score the run for quality, cost, and time |
| 7. The same question, from loaded files | Approve the waiting files, run the pipeline, and ask the leader's question |

Model requests in this notebook: **about 1**. A full run of this lab costs about 3 US cents.

## What this notebook is for

In notebook 4 a file waited for your approval for a few seconds. In real work it might wait all night. And while it waits, the program might stop: the computer restarts, the notebook is closed, something crashes.

In this notebook you make that happen on purpose. You pause a file, **restart the kernel**, and find the file still waiting. Then you prove the other half of the promise: after a restart, nothing is loaded twice.

A workflow that survives this is called **durable**.

> This notebook needs `OPENROUTER_API_KEY`. You will restart the kernel in the middle. That is part of the lab.

## Part 1: Before the crash

Run the cells of Part 1, then stop and follow the instructions in the box.

In [ ]:
from lab_helpers import LAB, reset_lab

from retail_profit_stack.agent import make_model
from retail_profit_stack.ingest.workflow import build_file_workflow, run_file

reset_lab()

In [ ]:
workflow = build_file_workflow(LAB, make_model(timeout=30))

for file_name in ["payment_fees_2026-08-31.csv", "orders_2026-09-14.csv", "payment_fees_2026-10-19.csv"]:
    result = run_file(workflow, file_name)
    print(f"{result['outcome']:8} {file_name}")

Two files loaded. One is waiting for approval, with the model's suggestion saved beside it.

> ### Now the crash
>
> **Restart the kernel now.** In VS Code, click **Restart** at the top of the notebook.
>
> Everything in the notebook's memory is gone: the workflow, the model, every variable.
>
> Then carry on from Part 2. **Do not run Part 1 again**: its first cell would clear the lab's folders.

## Part 2: After the crash

The new kernel knows nothing. We import again and build the workflow again. This time we give it no model, to show that none is needed to carry on.

In [ ]:
import pandas as pd
from lab_helpers import LAB

from retail_profit_stack.ingest import ledger
from retail_profit_stack.ingest.workflow import answer, build_file_workflow, run_file, waiting_files

workflow = build_file_workflow(LAB)

Ask what is waiting.

In [ ]:
for question in waiting_files(workflow, LAB):
    print("Waiting:  ", question["file_name"])
    print("Suggested:", question["suggested_mapping"])
    print("Reason:   ", question["suggestion"]["columns"][0]["reason"])

The file is still waiting, with the suggestion the model made **before** the restart. Nothing was asked again and nothing was paid for again. All of it came from the SQLite file.

Each file has its own thread, and its id is the file's name. That is how we can ask for one file's state.

In [ ]:
config = {"configurable": {"thread_id": "payment_fees_2026-10-19.csv"}}
saved = workflow.get_state(config)

print("Next node:", saved.next)
print("Missing:", saved.values["inspection"]["missing"], "  Unknown:", saved.values["inspection"]["unknown"])

Approve it, as if you had just come back to your desk.

In [ ]:
answer(workflow, "payment_fees_2026-10-19.csv", approved=True, person="your name")

## Part 3: The checkpoints

After every node, LangGraph wrote a **checkpoint**: a saved copy of the state. `get_state_history` lists a thread's checkpoints, newest first.

In [ ]:
for checkpoint in workflow.get_state_history(config):
    step = checkpoint.metadata["step"]
    print(f"step {step:>2}   next: {checkpoint.next!s:14} saved at {checkpoint.created_at[:19]}")

Read it from the bottom. The workflow started, ran `inspect`, then `suggest`, and stopped with `approval` next. Compare the time of that checkpoint with the one above it. The time between them is how long the file waited for you, across your restart. Then `approval`, `load`, and `finish` ran, and nothing is next.

A crash at any of these points would have been the same story: carry on from the last checkpoint.

## Part 4: Never load twice

After a crash, the node that was running is run **again**. So the `load` node might run twice for one file. If that loaded the rows twice, durability would have made things worse.

It does not, because of the load record from notebook 2. Run a loaded file again.

In [ ]:
result = run_file(workflow, "orders_2026-09-14.csv")

print(result["outcome"], "-", result["reason"])

And the resent copy, which has the same content under a new name.

In [ ]:
result = run_file(workflow, "orders_2026-09-14_resend.csv")

print(result["outcome"], "-", result["reason"])

Count the pieces on disk. A **piece** is one loaded file's rows. There should be one orders piece, however many times we tried.

In [ ]:
for piece in sorted(LAB.loaded.rglob("*.parquet")):
    print(piece.parent.name, "/", piece.name)

In [ ]:
pd.DataFrame(ledger.files_seen(LAB.state_file))[["file_name", "outcome", "rule", "rows_loaded", "approved_by"]]

Here is the line that does it, inside the `load` node. The fingerprint is claimed **before** the piece is written. The database lets only one file own a fingerprint.

In [ ]:
fingerprint = ledger.files_seen(LAB.state_file)[0]["fingerprint"]

print(ledger.claim_content(LAB.state_file, fingerprint, "some_other_name.csv"))

It returns the first owner, and the other name gets nothing.

## What we now know

- A **durable** workflow keeps its state on disk, so it survives a restart.
- After the restart, the waiting file was still there, with the model's suggestion. No model was called again.
- A **checkpoint** is written after every node. A workflow carries on from its last one.
- After a crash a node may run twice. So every node that changes something must be **safe to repeat**.
- Ours is safe because a fingerprint can be owned by one file only. The database enforces that, not our memory.

**Next:** notebook 6. One file at a time is slow. Fifty files, and some need a model. We load them side by side.